# 库、数据加载

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='TRUE'
os.environ['OMP_NUM_THREADS'] = '1'         # OpenMP
os.environ['MKL_NUM_THREADS'] = '1'         # Intel MKL
os.environ['OPENBLAS_NUM_THREADS'] = '1'    # OpenBLAS
os.environ['NUMEXPR_NUM_THREADS'] = '1'     # NumExpr
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'  # macOS Accelerate 框架

import torch
import numpy as np
import pandas as pd
import utils
from metrics import cal_clustering_metric
import scipy.io as scio
import random
import warnings
from sklearn.cluster import KMeans
import data_loader as loader
import time
warnings.filterwarnings('ignore') #忽略警告信息的输出


# 数据加载类

In [ ]:
class Dataset(torch.utils.data.Dataset): #Dataset是父类torch.utils.data.Dataset的子类，因此可以继承父类的属性
    #因为要训练自定义的数据，所以getitem和len是pytorch所提供的一种主要Map式数据集的必要改动。
    def __init__(self, X):
        self.X = X

    def __getitem__(self, idx):
        return self.X[:, idx], idx

    def __len__(self):
        return self.X.shape[1]

# 预训练类

In [ ]:
class PretrainDoubleLayer(torch.nn.Module):    
    #pretrain initialization
    def __init__(self, X, dim, device, act, batch_size=128, lr=10**-3): #self指代父类torch.nn.Module
        super(PretrainDoubleLayer, self).__init__() 
        self.X = X
        self.dim = dim
        self.lr = lr
        self.device = device
        self.enc = torch.nn.Linear(X.shape[0], self.dim) #全连接，输入层维度到表征维度
        self.dec = torch.nn.Linear(self.dim, X.shape[0]) #全连接，表征维度到输出层
        self.batch_size = batch_size
        self.act = act #activation function

 
        
    def forward(self, x):
        if self.act is not None: 
            z = self.act(self.enc(x))
            return z, self.act(self.dec(z))
        else:                    
            z = self.enc(x)
            return z, self.dec(z)

    def _build_loss(self, x, recons_x): 
        n = x.shape[0]
        return torch.norm(x-recons_x, p='fro')**2 / n #Frobenius范数且损失平均到每个样本

    def run(self):
        self.to(self.device) 
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr) 
        train_loader = torch.utils.data.DataLoader(Dataset(self.X), batch_size=self.batch_size, shuffle=True)

        loss = 0
        for epoch in range(10):
            for i, batch in enumerate(train_loader): #enumerate 索引和数据
                x, _ = batch 
                optimizer.zero_grad() 
                _, recons_x = self(x) 
                loss = self._build_loss(x, recons_x) #重构损失
                loss.backward() #梯度反向传播
                optimizer.step() #一步随机梯度下降
            print('epoch-{}: loss={}'.format(epoch, loss.item()))
        Z, _ = self(self.X.t()) 
        
        return Z.t() #返回表征空间

# 正式训练类

In [ ]:
class DKM(torch.nn.Module):
    def __init__(self,X,labels,layers=None,lr=10**-3,device=None, batch_size=256):
        super(DKM, self).__init__() #多重继承
        if layers is None:
            layers = [X.shape[0], 300, 80]
        if device is None:
            device = torch.device('cuda: 0' if torch.cuda.is_available() else 'cpu')
        self.layers = layers
        self.device = device
        if not isinstance(X, torch.Tensor): #检查数据是否是tensor形式
            X = torch.Tensor(X)
        self.X = X.to(device)
        self.labels = labels
        self.c = len(np.unique(labels)) #聚类个数c
        self.batch_size = batch_size
        self.lr = lr
        self._build_up()

    def _build_up(self): #不动
        self.act = torch.tanh
        self.enc1 = torch.nn.Linear(self.layers[0], self.layers[1]) 
        self.enc2 = torch.nn.Linear(self.layers[1], self.layers[2]) 
        self.dec1 = torch.nn.Linear(self.layers[2], self.layers[1]) 
        self.dec2 = torch.nn.Linear(self.layers[1], self.layers[0]) 

    def forward(self, x): #不动
        z = self.act(self.enc1(x)) 
        z = self.act(self.enc2(z)) 
        recons_x = self.act(self.dec1(z)) 
        recons_x = self.act(self.dec2(recons_x))
        return z, recons_x

    def _build_loss(self, z, x, d, recons_x):
        n = x.shape[0]
        loss = torch.norm(x - recons_x, p='fro') ** 2
        loss += np.nansum(d)
        loss1 = 0.0
        for i in range(z.shape[0]):
            center = self.centroids[self.labels[i]]
            loss1 += ((z[i] - center) ** 2).sum()
        return loss+loss1+0.0001 * (self.dec2.weight.norm()**2 + self.dec2.bias.norm()**2) / n
    
    def _update_D(self, Z):
        n = Z.shape[0]
        D = np.zeros((n, self.c))
        for j in range(self.c):
            D[:, j] = np.nansum((Z - np.tile(self.centroids[j, :], (n, 1))) ** 2, axis=1)
        return D

    
    def pretrain(self):
        string_template = '--------Start pretraining-{}--------'
        print(string_template.format(1))
        pre1 = PretrainDoubleLayer(self.X, self.layers[1], self.device, self.act, lr=self.lr)
        Z = pre1.run() #Pretrain的run
        self.enc1.weight = pre1.enc.weight
        self.enc1.bias = pre1.enc.bias
        self.dec2.weight = pre1.dec.weight
        self.dec2.bias = pre1.dec.bias
        print(string_template.format(2))
        pre2 = PretrainDoubleLayer(Z.detach(), self.layers[2], self.device, self.act, lr=self.lr)
        pre2.run()
        self.enc2.weight = pre2.enc.weight
        self.enc2.bias = pre2.enc.bias
        self.dec1.weight = pre2.dec.weight
        self.dec1.bias = pre2.dec.bias
        
    def run(self):
        self.pretrain()
        Z, _ = self(self.X.t()) 
        Z = Z.detach()
        
        idx = random.sample(list(range(Z.shape[0])), self.c)
        self.centroids = Z[idx,:]
        self.update_assign(Z)
        print('--------Start training-------------')
        train_loader = torch.utils.data.DataLoader(Dataset(self.X), batch_size=self.batch_size, shuffle=True)
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr)
        loss = 0
        for epoch in range(20):
            D = self._update_D(Z)
            for i, batch in enumerate(train_loader):
                x, idx = batch
                optimizer.zero_grad() #梯度清零
                z, recons_x = self(x)
                d = D[idx, :]
                loss = self._build_loss(z, x, d, recons_x)
                loss.backward()
                optimizer.step() #更新模型参数 
            Z, _ = self(self.X.t())
            Z = Z.detach()
            
            self.clustering(Z) #更新类中心和指派
            nmi, ari,acc,ri = cal_clustering_metric(self.labels, self.pre_labels)
            print('epoch-{}, NMI={}, ARI={},ACC={},RI={}'.format(epoch, nmi, ari,acc,ri))
    
    def clustering(self, Z):
        self.updata_v(Z)
        self.update_assign(Z)
    
    def update_assign(self, Z):
        n = Z.shape[0]
        distances = np.linalg.norm(Z[:, np.newaxis, :] - self.centroids[np.newaxis, :, :], axis=2)
    
        # 取最小距离对应的簇索引
        pre_labels = np.argmin(distances, axis=1)
        self.pre_labels = pre_labels
    
    def updata_v(self, Z):  # Z 是 torch.Tensor，形状 [n_samples, n_features]
        n_features = Z.shape[1]
        device = Z.device

        centroids = torch.zeros((self.c, n_features), device=device)

        for k in range(self.c):
            cluster_points = Z[self.pre_labels == k]
            if cluster_points.shape[0] > 0:
                centroids[k] = cluster_points.mean(dim=0)
            else:
                rand_idx = torch.randint(0, Z.shape[0], (1,))
                centroids[k] = Z[rand_idx]

        self.centroids = centroids  # 保持是 torch.Tensor
        

# Experiments

In [ ]:
#else
dkm = DKM(data, labels,batch_size=512, lr=1e-4)
dkm.run()